In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

%matplotlib widget

# Path to the in-loop tracking H5 output.
# For your own run: point this to the .h5 produced by Force_Minimizer with track_axis=true.
# The example file ships with the notebook but is excluded from git (too large).
H5_FILE = "ciematqi4x_10vol_lcons3_opt_opt.h5"


# In-loop fixed-point tracking — reading results from a SPECTRE optimisation run

The `FixedPointTracker` runs inside the optimiser loop (via `Force_Minimizer` with `track_axis = true` or `track_islands = true` in the TOML).
After each accepted iteration it writes updated results to the `/iterations/fp` group in the H5 output file.

This notebook loads that group and plots how the magnetic axis and a 8/9 island chain evolved
over 52 optimisation iterations of a CIEMAT QI4X 10-volume equilibrium.


In [ ]:
with h5py.File(H5_FILE, "r") as f:
    fp = f["iterations/fp"]
    axis_rz = fp["axis_rz"][:]
    island_rz = fp["island_rz"][:]
    island_rz_hp = fp["island_rz_half_period"][:]
    residual = fp["residual"][:]
    iota = fp["iota"][:]
    iota_rel = fp["iota_rel"][:]
    iteration_indices = fp["iteration_indices"][:].astype(int)

# Trim to valid (non-NaN) entries — pre-allocated arrays are filled up to n_done.
valid = ~np.isnan(axis_rz[:, 0])
n = valid.sum()
iters = iteration_indices[valid]
axis_rz = axis_rz[valid]
island_rz = island_rz[valid]
island_rz_hp = island_rz_hp[valid]
residual = residual[valid]
iota = iota[valid]
iota_rel = iota_rel[valid]

print(f"Loaded {n} iterations (iter {iters[0]} .. {iters[-1]})")
print(f"island_rz  R: {island_rz[:, 0].min():.4f} .. {island_rz[:, 0].max():.4f}")
print(f"residual     : {residual.min():.1e} .. {residual.max():.1e}")
print(f"iota         : {iota.min():.6f} .. {iota.max():.6f}  (target: 8/9 = {8/9:.6f})")


## 1 — Magnetic axis position over optimisation iterations

The axis drifts inward (decreasing R) as the equilibrium is optimised. Z stays near 0 throughout.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(iters, axis_rz[:, 0], "o-", ms=3, lw=1)
axes[0].set_xlabel("Optimisation iteration")
axes[0].set_ylabel("R [m]")
axes[0].set_title("Magnetic axis R")

axes[1].plot(iters, axis_rz[:, 1], "o-", ms=3, lw=1, color="tab:orange")
axes[1].set_xlabel("Optimisation iteration")
axes[1].set_ylabel("Z [m]")
axes[1].set_title("Magnetic axis Z")

fig.tight_layout()
plt.show()


## 2 — Island O-point and half-period position over iterations

The O-point (island centre) and its half-period companion are tracked each iteration.
`island_rz` is the O-point; `island_rz_half_period` is the point at the toroidal midpoint
between two consecutive O-points (half a poloidal revolution around the chain).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for col, (label, rz) in enumerate([("O-point", island_rz), ("half-period", island_rz_hp)]):
    axes[0, col].plot(iters, rz[:, 0], "o-", ms=3, lw=1)
    axes[0, col].set_xlabel("Optimisation iteration")
    axes[0, col].set_ylabel("R [m]")
    axes[0, col].set_title(f"Island {label} R")

    axes[1, col].plot(iters, rz[:, 1], "o-", ms=3, lw=1, color="tab:orange")
    axes[1, col].set_xlabel("Optimisation iteration")
    axes[1, col].set_ylabel("Z [m]")
    axes[1, col].set_title(f"Island {label} Z")

fig.tight_layout()
plt.show()

# Greene's residue R = (2 - tr) / 4; R < 0 → X (hyperbolic), R in (0, 0.25) → O (elliptic).
fig2, ax2 = plt.subplots(figsize=(8, 3.5))
ax2.semilogy(iters, np.abs(residual), "o-", ms=3, lw=1, color="tab:green", label="|residue|")
ax2.set_xlabel("Optimisation iteration")
ax2.set_ylabel("|Greene residue|")
ax2.set_title("Greene residue magnitude — O-point (smaller is more elliptic / narrower island)")
ax2.legend()
fig2.tight_layout()
plt.show()

print(f"Final residue: {residual[-1]:.2e}  (|residue| << 0.25 => strongly elliptic O-point)")
print(f"Final iota:    {iota[-1]:.6f}  (target 8/9 = {8/9:.6f}, rel error {iota_rel[-1]:.1e})")
